# 02 — Preprocessing

Parse dates, confirm there are no missing values, and apply a **temporal** train/test split (most recent 20% of dates). Random row-wise splits leak the future into the past.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data_loader import load_dataset
from src.feature_engineer import engineer_features
from src.preprocessor import FeatureEncoder, temporal_split

df = load_dataset()
print(df.dtypes)
print("nulls", int(df.isna().sum().sum()))


One-hot encoding and scaling are **fit on train only**. The `FeatureEncoder` does that. Trees do not need scaled inputs; linear models do (`scale=True`).


In [ ]:
featured = engineer_features(df)
split = temporal_split(featured)
print("cutoff", split.cutoff.date(), "train", len(split.train), "test", len(split.test))
assert split.train["Date"].max() < split.test["Date"].min()

enc = FeatureEncoder("operational").fit(split.train)
X_train = enc.transform(split.train, scale=False)
X_train_scaled = enc.transform(split.train, scale=True)
X_test = enc.transform(split.test, scale=False)
print(X_train.shape, X_test.shape)
X_train.head()


Columns **not** used as operational predictors:

- `Units Sold` (target)
- `Units Ordered` (replenishment decision, ~0 correlation with sales)
- `Demand Forecast` (reserved for the vendor-refinement track)
